![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Text-to-Speech, Voice Cloning, and the Anatomy of AI Slop

Last lab we turned speech into text. This one runs the arrow backwards, and then does something slightly unwise with the result.

1. **Voice cloning.** Two models — Qwen3-TTS and F5-TTS — cloning the same voice from the same reference clips, saying the same sentence. Same inputs, different architectures, audibly different outputs.
2. **The slop video.** A full content-farm pipeline: an LLM writes a Reddit story, a cloned voice narrates it, Whisper times every word, ffmpeg staples it to gameplay footage, and a caption renderer bounces the words on screen. Four models and one ffmpeg call is genuinely all it takes.

**Data:** three clips of Yae Miko (Genshin Impact) as cloning references, and 107 MB of Subway Surfers footage as a background. I'm not pretending that's a neutral choice of material. The genre *is* the lesson.

**The link to the previous lab is not decorative.** TTS voice cloning needs a reference clip *and its transcript*. We don't type the transcripts — Whisper produces them. Part 2 then feeds the TTS output back into Whisper to measure how intelligible it is. STT and TTS are two ends of the same pipe.

---

### Before you run anything

1. **Enable a GPU.** Runtime → Change runtime type → GPU. I haven't tried it, tho it 99% works on a free T4. as I maintaind VRAM and models are small, however I ran it on an A100.
2. **Have a Hugging Face token ready.** Everything in this notebook is open weights running on this machine. No model is gated, and no cell calls a hosted inference API — not for TTS, not for captions, not for anything.
3. **Run it in order.** Sections free the GPU on the way out; skipping around will hand you an OOM you didn't earn.

In [ ]:
%pip install -q \
    qwen-tts \
    f5-tts \
    faster-whisper==1.2.1 \
    ctranslate2==4.8.1 \
    soundfile==0.13.1 \
    librosa==0.11.0 \
    srt==3.5.3 \
    beautiful-captions==0.1.71

### 0.1 — Verify the environment

Same as last time: if the installs downgraded each other into an incompatible set, this cell fails in two seconds instead of forty minutes deep into a video render.

It also refuses to continue without a GPU. Every stage below is CUDA-only, and Qwen3-TTS in particular has no CPU path.

In [ ]:
import numpy, torch, transformers, librosa, soundfile, srt, ctranslate2
from faster_whisper import WhisperModel

print(f"numpy        {numpy.__version__}")
print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"librosa      {librosa.__version__}")
print(f"ctranslate2  {ctranslate2.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    raise RuntimeError("No GPU detected. Runtime -> Change runtime type -> GPU.")
print("\nEnvironment OK")

### 0.2 — Hugging Face authentication

Nothing in this lab is gated, but the dataset repo and the model weights all come off the Hub and an authenticated session downloads faster and rate-limits less. This is a **Hub token, not an inference API key**. Everything runs here.

In [ ]:
from huggingface_hub import login, whoami

login()  # paste a token with 'read' scope

print(f"Logged in as: {whoami()['name']}")

### 0.3 — Configuration

Every tunable lives here. Nothing below this cell hardcodes a value.

The one worth looking at twice is `REF_WINDOWS`. The Yae Miko mp3s are longer than a reference clip should be, so each one gets a `(start, duration)` window in seconds. Listen to the trimmed clips in section 1, then come back here and move the windows if a clip starts mid-word or lands on background music.

In [ ]:
import os
from pathlib import Path

# --- models
WHISPER_MODEL  = "large-v3"
QWEN_TTS_MODEL = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
SCRIPT_MODEL   = "Qwen/Qwen3-4B-Instruct-2507"
# SCRIPT_MODEL = "Qwen/Qwen3-14B"   # alternative: hybrid-thinking model, so pass
#                                   # enable_thinking=False to apply_chat_template or you
#                                   # will parse a <think> block as JSON. ~28 GB in bf16.

# --- data
REPO            = "frost000/TTS_LAB"
REF_FILES       = ["yae_miko_1.mp3", "yae_miko_2.mp3", "yae_miko_3.mp3"]
BACKGROUND_FILE = "subway_surfers.mp4"

# --- reference windows: filename -> (start_seconds, duration_seconds)
REF_WINDOWS = {
    "yae_miko_1.mp3": (0.0, 10.0),
    "yae_miko_2.mp3": (0.0, 10.0),
    "yae_miko_3.mp3": (0.0, 10.0),
}
NARRATOR_FILE   = "yae_miko_1.mp3"   # which reference narrates Part 2
MAX_REF_SECONDS = 12.0               # cap applied to an uploaded clip
USER_REF_TEXT   = "Most humans could have the entirety of their fleeting lives chronicled in but a few short lines... Their stories really make for terribly dull reading."

# --- audio
TARGET_SR       = 16_000    # what Whisper wants; the TTS models resample internally
SENTENCE_GAP_MS = 120       # silence stitched between narrated sentences

# --- the sentence every clone says, so the only variable is the reference
CLONE_TEXT = ("I have been listening to your conversation for some time now, "
              "and I must say, the entertainment value is considerable.")

# --- three sentences for the stable-voice demo in 1.2
DEMO_SENTENCES = [
    "The shrine is quiet this evening, which is unusual and therefore interesting.",
    "You should know that I remember every favour, and I collect on all of them.",
    "Do continue. I am enjoying this far more than you appear to be.",
]

# --- script generation
MAX_SENTENCES    = 9
TARGET_WORDS     = 150
WORDS_PER_MINUTE = 150
TEMPERATURE      = 0.9
TOP_P            = 0.95
MAX_NEW_TOKENS   = 1024

# --- video
VIDEO_W, VIDEO_H = 1080, 1920
CRF              = 23
PRESET           = "veryfast"
AUDIO_BITRATE    = "192k"

# --- captions (beautiful-captions)
# 'font' is a FILE STEM, not a font name. Valid values are exactly:
#   CheGuevaraBarry-Brown, FiraSansCondensed-ExtraBoldItalic, Gabarito-Black,
#   KOMIKAX_, Montserrat-Bold, Proxima-Nova-Semibold, Rubik-ExtraBold
CAPTION_FONT       = "KOMIKAX_"   # Komika Axis, the MrBeast font. Correct for the genre.
# CAPTION_FONT     = "Montserrat-Bold"   # the tasteful alternative
CAPTION_FONT_SIZE  = 140
CAPTION_COLOR      = "white"
CAPTION_OUTLINE    = "black"
CAPTION_THICKNESS  = 12
CAPTION_POSITION   = 0.5          # their spelling is 'verticle'; 0.0 bottom -> 1.0 top
MAX_WORDS_PER_LINE = 1
CAPTION_ANIMATION  = "bounce"
CAPTION_KEYFRAMES  = 10

# --- reproducibility
SEED = 0   # also passed to F5-TTS, where a missing seed is actively dangerous. See 1.4.

# --- output
REF_DIR    = Path("/content/refs")
OUTPUT_DIR = Path("/content/outputs")
for d in (REF_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

NARRATION = OUTPUT_DIR / "narration.wav"
SRT_PATH  = OUTPUT_DIR / "captions.srt"
COMPOSITE = OUTPUT_DIR / "composite.mp4"
FINAL     = OUTPUT_DIR / "slop.mp4"
FRAME     = OUTPUT_DIR / "slop_frame.jpg"

os.environ.setdefault("HF_HOME", "/content/hf_cache")

print(f"References -> {REF_DIR}")
print(f"Outputs    -> {OUTPUT_DIR}")

### 0.4 — Small helpers

The timer and the VRAM release from last lab, unchanged except that `stage()` now accumulates into `TIMINGS`. We spend that dictionary at the end of the notebook.

Two new ones:

- `run()` shells out and raises with the tail of stderr attached. ffmpeg failures are unreadable if you only get a return code.
- `sanitize_env()` cleans up after F5-TTS. Section 1.4 explains why that sentence is necessary.

In [ ]:
  import gc, subprocess, time
from contextlib import contextmanager

import torch

TIMINGS = {}


@contextmanager
def stage(name: str):
    '''Time a stage, print how long it took, and record it in TIMINGS.'''
    print(f"[{name}] started")
    t0 = time.time()
    yield
    elapsed = time.time() - t0
    TIMINGS[name] = TIMINGS.get(name, 0.0) + elapsed
    print(f"[{name}] finished in {elapsed:.1f}s\n")


def free_vram(*objects) -> None:
    '''Delete model objects and return their memory to the GPU.'''
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()
    used = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM in use after cleanup: {used:.2f} GB")


def run(cmd) -> str:
    '''Run a command, raising with stderr attached if it fails.'''
    result = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"{cmd[0]} failed ({result.returncode}):\n{result.stderr[-2000:]}")
    return result.stdout.strip()


def sanitize_env():
    '''F5-TTS can set PYTHONHASHSEED outside its legal range, which kills every
    Python subprocess launched afterwards. Belt and braces alongside seed=.'''
    seed = os.environ.get("PYTHONHASHSEED")
    if seed is not None and not (seed == "random" or (seed.isdigit() and int(seed) <= 4294967295)):
        print(f"PYTHONHASHSEED was {seed!r} - out of range, removing it")
        del os.environ["PYTHONHASHSEED"]

### 0.5 — Patch beautiful-captions

Two chores that have to happen before anything imports `beautiful_captions`, which is why they're up here at the top rather than buried in Part 2 where the library is actually used.

**The imports are broken.** Version 0.1.71 ships two files that do `from src.beautiful_captions.core.types import ...`. That path only resolves if you're running from a checkout of the source tree; from an installed wheel, `import beautiful_captions` raises `ModuleNotFoundError: No module named 'src'`. The `sed` below rewrites those imports in place and clears the bytecode cache so the fix actually takes.

**libass can't find the fonts.** The package bundles its .ttf files inside site-packages, where the system font config never looks. We copy them into `/usr/share/fonts` and rebuild the cache. Skip this and every caption renders in DejaVu Sans, silently, with no error anywhere.

This is what "works locally" looks like when it reaches you as a wheel.

In [ ]:
import importlib.util, os

BC_DIR = os.path.dirname(importlib.util.find_spec("beautiful_captions").origin)
print(f"beautiful-captions installed at: {BC_DIR}")

# a) fix the 'from src.beautiful_captions' imports
!grep -rl "src\.beautiful_captions" {BC_DIR} --include="*.py" | xargs -r sed -i 's/from src\.beautiful_captions/from beautiful_captions/g'
!find {BC_DIR} -name "__pycache__" -type d -exec rm -rf {{}} + 2>/dev/null

# b) make the bundled fonts visible to libass
!mkdir -p /usr/share/fonts/truetype/bcaptions
!cp {BC_DIR}/fonts/*.ttf /usr/share/fonts/truetype/bcaptions/
!fc-cache -f > /dev/null 2>&1

print("Fonts installed:")
!ls /usr/share/fonts/truetype/bcaptions/

## 1 — Voice cloning

### 1.0 — Load and trim the references

Same pattern as last lab: `hf_hub_download` gives us a local path, `librosa.load` decodes to mono float32 at 16 kHz.

The difference is what we do next. A cloning reference is not a recording it's a *specification of a voice*, and longer is not better. Ten seconds of clean, well-articulated speech beats a minute that wanders through background music and a cough. Both models condition on the whole clip, so anything in there is something they might reproduce.

So we slice each file to a predefined amount of time, and write the result to disk as wav, because F5-TTS takes file paths, not arrays.

In [ ]:
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
import librosa, soundfile as sf

ref_paths = {}
with stage("download + trim references"):
    for name in REF_FILES:
        src = hf_hub_download(REPO, name, repo_type="dataset")
        audio, _ = librosa.load(src, sr=TARGET_SR, mono=True)

        start_s, dur_s = REF_WINDOWS[name]
        clip = audio[int(start_s * TARGET_SR): int((start_s + dur_s) * TARGET_SR)]

        out = REF_DIR / f"{Path(name).stem}.wav"
        sf.write(out, clip, TARGET_SR)
        ref_paths[name] = out

        print(f"{name}: full {len(audio)/TARGET_SR:5.1f}s  ->  "
              f"trimmed {len(clip)/TARGET_SR:4.1f}s  [{start_s:.1f}s +{dur_s:.1f}s]")

background_path = hf_hub_download(REPO, BACKGROUND_FILE, repo_type="dataset")
print(f"\nbackground: {background_path}")

In [ ]:
for name, path in ref_paths.items():
    print(f"\n{name}  ->  {path.name}")
    display(Audio(str(path)))

### 1.1 — The reference transcripts come from Whisper, not from me

Both TTS models want a pair: `(reference_audio, reference_transcript)`. The transcript is not a label for our benefit — it tells the model which acoustics correspond to which phonemes, which is how it separates *this voice* from *these words*.

I could type the three transcripts out. I'm not going to, and the reason is the whole point of running these two labs back to back: **the previous lab produces exactly the input this one requires.** STT feeds TTS. Whisper large-v3 transcribes the trimmed clips, and its output becomes `ref_text`.

It also means Whisper's mistakes are now upstream of everything. If it mishears a word in the reference, the cloning model is conditioned on a lie about what was said. Read the transcripts below against what you heard.

In [ ]:
from faster_whisper import WhisperModel

with stage("load faster-whisper large-v3"):
    whisper = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="float16")

ref_texts = {}
with stage("transcribe references"):
    for name, path in ref_paths.items():
        segments, info = whisper.transcribe(str(path), language="en", beam_size=5)
        ref_texts[name] = " ".join(seg.text.strip() for seg in segments).strip()

for name, text in ref_texts.items():
    print(f"{name}\n  {text}\n")

In [ ]:
free_vram(whisper)

### 1.2 — Qwen3-TTS

`Qwen3-TTS-12Hz-1.7B-Base` is the voice-clone member of the Qwen3-TTS family: Apache 2.0, ungated, about 4.5 GB in bf16. Architecturally it's an **autoregressive language model over discrete speech tokens** — a 12 Hz multi-codebook tokenizer turns audio into something that looks enough like text for a transformer to model, and generation is next-token prediction all the way down. That framing matters later, when we compare it to something built on a completely different principle.

One deliberate choice in the load: `attn_implementation="sdpa"` rather than `flash_attention_2`. FlashAttention is faster, but installing it in Colab means a twenty-minute source build and an Ampere-or-newer card. PyTorch's built-in scaled-dot-product attention is right there, already compiled, and fast enough that you'd need a stopwatch to care.

In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

with stage("load Qwen3-TTS"):
    qwen_tts = Qwen3TTSModel.from_pretrained(
        QWEN_TTS_MODEL,
        device_map="cuda:0",
        dtype=torch.bfloat16,
        attn_implementation="sdpa",   # NOT flash_attention_2 - 20min build, needs Ampere+
    )

print("Loaded.")

**The experiment:** one model, one sentence, three references. Everything is held fixed except which ten seconds of Yae Miko the model was shown.

**reference quality dominates output quality.** People reach for a bigger model when sometimes the fix is a better ten seconds of audio.

In [ ]:
clone_outputs = {}
with stage("clone from each reference"):
    for name, path in ref_paths.items():
        wavs, sr = qwen_tts.generate_voice_clone(
            text=CLONE_TEXT,
            language="English",
            ref_audio=str(path),
            ref_text=ref_texts[name],
        )
        out = OUTPUT_DIR / f"qwen_clone_{Path(name).stem}.wav"
        sf.write(out, wavs[0], sr)
        clone_outputs[name] = out

print(f'Target sentence: "{CLONE_TEXT}"\n')
for name, out in clone_outputs.items():
    print(f"--- from {name} ---")
    print(f"    ref_text: {ref_texts[name][:90]}...")
    display(Audio(str(out)))

**Keeping a voice stable across a script.**

Calling `generate_voice_clone` once per sentence re-derives the speaker conditioning from the reference every time. Sampling is stochastic, so you get a slightly different reading of the same voice on each call, and across a dozen sentences that drift is audible — the narrator ages, or changes their mind about how much energy they have.

`create_voice_clone_prompt` does the conditioning **once** and returns a reusable prompt object. Every sentence generated against it starts from the same speaker state, so the voice holds. It's also faster, since the reference is only encoded once.

Part 2 depends on this. A sixty-second narration is nine or ten sentences, and nine independently-cloned readings sound like nine different people doing an impression of the same character.

In [ ]:
with stage("stable multi-sentence clone"):
    prompt_items = qwen_tts.create_voice_clone_prompt(
        ref_audio=str(ref_paths[NARRATOR_FILE]),
        ref_text=ref_texts[NARRATOR_FILE],
    )
    wavs, sr = qwen_tts.generate_voice_clone(
        text=DEMO_SENTENCES,
        language=["English"] * len(DEMO_SENTENCES),
        voice_clone_prompt=prompt_items,
    )

for sentence, wav in zip(DEMO_SENTENCES, wavs):
    print(sentence)
    display(Audio(wav, rate=sr))

### 1.3 — Your turn: clone something if ya want

Upload a voice clip, ten to twenty seconds of talking normally, not reading dramatically, and put its transcript in `USER_REF_TEXT` in the config cell.

In [ ]:
from google.colab import files

uploaded = files.upload()
user_clip = next(iter(uploaded))

user_audio, _ = librosa.load(user_clip, sr=TARGET_SR, mono=True)
user_audio = user_audio[: int(MAX_REF_SECONDS * TARGET_SR)]
user_ref = REF_DIR / "user_reference.wav"
sf.write(user_ref, user_audio, TARGET_SR)
print(f"{user_clip}: kept {len(user_audio)/TARGET_SR:.1f}s -> {user_ref}")

In [ ]:
with stage("clone the uploaded voice"):
    wavs, sr = qwen_tts.generate_voice_clone(
        text=CLONE_TEXT,
        language="English",
        ref_audio=str(user_ref),
        ref_text=USER_REF_TEXT,
    )

sf.write(OUTPUT_DIR / "qwen_clone_user.wav", wavs[0], sr)
print("Reference:")
display(Audio(str(user_ref)))
print("Clone:")
display(Audio(wavs[0], rate=sr))

### 1.4 — F5-TTS: the architectural contrast

F5-TTS solves the same problem from the opposite direction. Where Qwen3-TTS generates speech tokens one at a time, F5-TTS is a **non-autoregressive flow-matching Diffusion Transformer** with ConvNeXt V2 blocks over the text stream: it starts from noise shaped like the whole output and refines it in parallel, in a fixed number of steps, with no left-to-right dependency at all.

Two models, one task, opposite mechanisms. That's the reason to run both rather than picking a winner.

Qwen3-TTS goes first out of VRAM.

In [ ]:
free_vram(qwen_tts)

#### The `seed=None` landmine
(you don't need to understand this, it's just model specfic peculiarity)
If you call `F5TTS.infer()` without a `seed`, the library does this:

```python
seed = random.randint(0, sys.maxsize)
os.environ["PYTHONHASHSEED"] = str(seed)
```

`sys.maxsize` is 9223372036854775807. The legal range for `PYTHONHASHSEED` is `[0, 4294967295]`. So roughly always, the process environment now contains a value CPython rejects — and every Python subprocess launched afterwards dies at interpreter startup with `Fatal Python error: config_init_hash_seed`. That breaks `pip`. It breaks anything that shells out to Python. For the rest of the session. From a text-to-speech call.

Nothing about this is exotic. A library set a **global** — an environment variable, inherited by every child process — as a side effect of a function that had no business touching the environment at all, and the failure surfaces somewhere completely unrelated, later, in a traceback that mentions nothing about TTS. When you write library code, the blast radius of a global is the whole process tree.

Two defences, and we use both:

1. Pass an explicit `seed` to `infer()`. This is good practice anyway — it makes the output reproducible.
2. Call `sanitize_env()` afterwards, which drops the variable if it's out of range.

It cost me around 3 hours of debugging

In [ ]:
from f5_tts.api import F5TTS

F5_OUT = OUTPUT_DIR / "f5_clone.wav"

with stage("load + run F5-TTS"):
    f5 = F5TTS()
    f5.infer(
        ref_file=str(ref_paths[NARRATOR_FILE]),
        ref_text=ref_texts[NARRATOR_FILE],
        gen_text=CLONE_TEXT,
        file_wave=str(F5_OUT),
        seed=SEED,          # MANDATORY - see the markdown above
    )

display(Audio(str(F5_OUT)))

In [ ]:
free_vram(f5)
sanitize_env()
print(f"PYTHONHASHSEED is now: {os.environ.get('PYTHONHASHSEED')!r}")

#### Side by side

Reference, then Qwen3-TTS, then F5-TTS. Same reference clip, same sentence, same seed where a seed exists.

In [ ]:
print("=" * 50)
print("REFERENCE  (Yae Miko, trimmed)")
print("=" * 50)
display(Audio(str(ref_paths[NARRATOR_FILE])))

print("=" * 50)
print("QWEN3-TTS  (autoregressive speech-token LM)")
print("=" * 50)
display(Audio(str(clone_outputs[NARRATOR_FILE])))

print("=" * 50)
print("F5-TTS  (flow-matching DiT)")
print("=" * 50)
display(Audio(str(F5_OUT)))

#### What actually differs

**Architecture.** Qwen3-TTS is an autoregressive LM over multi-codebook speech tokens at 12 Hz. F5-TTS is a non-autoregressive flow-matching Diffusion Transformer with ConvNeXt V2. Autoregression tends to produce more natural long-range prosody, because each token sees everything before it; flow matching generates the whole utterance in parallel, which is fast but has to commit to the global shape up front.

**Language coverage.** Qwen3-TTS covers ten languages. The F5-TTS base checkpoint is English and Chinese. If your script isn't in one of those two, the comparison ends there.

**Duration.** F5-TTS caps at **30 seconds total — reference plus generated, together.** A 10-second reference leaves you 20 seconds of output. This is a hard architectural limit, not a quality suggestion.

That limit is why **Part 2 narrates with Qwen3-TTS**, and I want that on the record as a decision rather than an omission. A 60-second narration through F5-TTS would mean chunking the script, running each chunk against the reference separately, and stitching — which reintroduces exactly the voice drift that `create_voice_clone_prompt` exists to prevent.

---

## 2 — The AI slop video

What follows is a parody of a real pipeline. Content farms are running some version of this right now, at volume, monetised.

The uncomfortable part isn't that it exists. It's how little there is to it. An LLM writes the story, a TTS model reads it in a cloned voice, an ASR model times the words, ffmpeg staples it over gameplay footage, and a caption library bounces the text. **Four models and one ffmpeg call.** No secret sauce, no proprietary anything, nothing here that wasn't free.

### 2.1 — The script

`Qwen3-4B-Instruct-2507` again, same model as the translation stage last lab. It's a non-thinking instruct model, so there's no `<think>` block to strip before the JSON parse. (The commented alternative in the config cell, `Qwen3-14B`, *is* hybrid-thinking — it needs `enable_thinking=False` and about 28 GB in bf16.)

**The arithmetic.** Target is 60–75 seconds. English narration lands around 150 words per minute, so ~150 words ≈ 60 s. At roughly 15–18 words a sentence that's 9 sentences, which is why `MAX_SENTENCES = 9`. We don't trim afterwards — we ask for the right length up front and check the narration duration in 2.2.

**Where the quality actually comes from.** Not the model. The few-shot examples. Three complete stories, written out in full below, with the exact voice and structure we want. A 4B model with three strong examples beats a 30B model with none, every time, and the reason is that "write a viral Reddit story" is a style instruction the model can't ground, while three examples *are* the grounding.

The instructions demand four things, in order of how much they matter: a first sentence that opens a loop the reader needs closed, escalating stakes, one specific concrete detail early (a number, a name, a brand — specificity is what makes fiction feel reported), and a last line that either lands or hands the judgement to the audience.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

with stage("load script LLM"):
    tokenizer = AutoTokenizer.from_pretrained(SCRIPT_MODEL)
    llm = AutoModelForCausalLM.from_pretrained(
        SCRIPT_MODEL,
        dtype=torch.bfloat16,
        device_map="auto",
    )

print(f"Parameters: {llm.num_parameters() / 1e9:.2f}B")

In [ ]:
import json, re

SYSTEM_PROMPT = (
    "You write short first-person stories for narrated social video — the ones that play "
    "over gameplay footage with bouncing captions. Your voice is conversational and "
    "present-tense, like someone telling a friend something that just happened to them. "
    "You are funny, specific, and slightly aggrieved."
)

FEWSHOT = [
    {
        "title": "My roommate has been charging me rent on an apartment he doesn't pay for",
        "sentences": [
            "So I just found out my roommate has been charging me rent for eight months on an apartment he doesn't actually pay for.",
            "His uncle owns the building and lets him live there for free.",
            "I only found out because the uncle turned up to fix the sink and asked me who I was.",
            "Six hundred dollars a month, every month, into a Venmo account he labelled 'utilities'.",
            "I haven't said anything yet, because I want to see how long he keeps going.",
            "He asked me last night if I could pay early this month.",
            "What should i do?",
        ],
    },
    {
        "title": "I got banned from the book club I started",
        "sentences": [
            "I have been banned from my own book club by the people I invited to it.",
            "It started when I suggested we should maybe read the book before the meeting.",
            "Apparently that was 'creating a hostile environment', which is a bold thing to say about a group discussing a novel set in a lighthouse.",
            "There was a vote. I lost seven to one, and the one was me.",
            "They kept the name, the group chat, and the snack rotation spreadsheet I built.",
            "Luckily I own the meeting place. at their second meeting I had the security kick them out of the building.",
            "Was I wrong for Kicking them out?"
        ],
    },
    {
        "title": "My sister invited me to her wedding as staff",
        "sentences": [
            "My sister invited me to her wedding, but not as a guest — as the photographer, unpaid.",
            "The invitation arrived as a Google Calendar event with the word 'shift' in the title.",
            "I have shot exactly one wedding in my life, and it was hers, four years ago, before the first divorce.",
            "When I asked whether I could at least sit at a table, she said the seating chart was already printed.",
            "So I said yes I'll be there, count me in !",
            "and then I booked a flight to Lisbon for that weekend.",
            "I am going to let you decide whether that makes me the villain.",
        ],
    },
]

INSTRUCTIONS = """Write one original story in the same voice as the examples.

Requirements:
- The first sentence must open a loop the audience needs closed. No preamble, no scene-setting.
- Stakes escalate. Every sentence is worse or stranger than the one before it.
- Include one specific concrete detail early — a number, a brand, a name, an amount of money.
- The last line either hands the judgement to the audience.
- Between 6 and {max_sentences} sentences. Aim for about {target_words} words total.
- No semicolons, no lists, no parentheses.
- Do not reuse any situation from the examples.

Return ONLY a JSON object with keys "title" (string) and "sentences" (array of strings).
No markdown fences, no commentary."""


def extract_json_object(raw: str) -> dict:
    """Pull the first JSON object out of a model response."""
    match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if match is None:
        raise ValueError(f"No JSON object found in model output:\n{raw[:400]}")
    return json.loads(match.group(0))


messages = [{"role": "system", "content": SYSTEM_PROMPT}]
for example in FEWSHOT:
    messages.append({"role": "user", "content": "Write a story."})
    messages.append({"role": "assistant",
                     "content": json.dumps(example, ensure_ascii=False)})
messages.append({"role": "user", "content": INSTRUCTIONS.format(
    max_sentences=MAX_SENTENCES, target_words=TARGET_WORDS)})

print(f"{len(messages)} messages in the prompt "
      f"({len(FEWSHOT)} few-shot examples + instructions)")

In [ ]:
import json # Ensure json is imported if not already

MAX_RETRIES = 3 # Define max retries for robustness

with stage("generate script"):
    script = None # Initialize script to None
    for attempt in range(MAX_RETRIES):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(llm.device)

        generated = llm.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.eos_token_id,
        )
        reply = tokenizer.decode(generated[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        try:
            script = extract_json_object(reply)
            # If successful, break the loop
            break
        except json.JSONDecodeError as e:
            print(f"Attempt {attempt + 1}/{MAX_RETRIES}: JSON parsing failed. Error: {e}")
            print(f"Problematic reply (first 500 chars):\n{reply[:500]}...")
            if attempt == MAX_RETRIES - 1:
                print("All retries failed. Re-raising the error.")
                raise # Re-raise if all retries fail
        except ValueError as e: # Catch the ValueError from extract_json_object if no JSON is found
            print(f"Attempt {attempt + 1}/{MAX_RETRIES}: No JSON object found. Error: {e}")
            print(f"Problematic reply (first 500 chars):\n{reply[:500]}...")
            if attempt == MAX_RETRIES - 1:
                print("All retries failed. Re-raising the error.")
                raise # Re-raise if all retries fail

    if script is None:
        raise RuntimeError("Failed to generate a valid script after multiple retries.")

sentences = script["sentences"][:MAX_SENTENCES]
word_count = sum(len(s.split()) for s in sentences)

print(f"TITLE: {script['title']}\n")
for i, sentence in enumerate(sentences, 1):
    print(f"{i:>2}. {sentence}")
print(f"\n{len(sentences)} sentences, {word_count} words "
      f"-> ~{word_count / WORDS_PER_MINUTE * 60:.0f}s at {WORDS_PER_MINUTE} wpm")

In [ ]:
free_vram(llm)

### 2.2 — Narration

Qwen3-TTS comes back, and this is where `create_voice_clone_prompt` earns its place: **one** clone prompt from the chosen Yae Miko reference, then every sentence generated against it in a single batched call. One speaker state, one voice, nine sentences.

We stitch them with 120 ms of silence between sentences. Without it the sentences butt up against each other and the narration sounds like someone reading a list. With much more than that it sounds like a hostage video. This number is a taste parameter and it's in the config cell for a reason.

of course usually people clone specific people depending on the script ..
for example some more "philosophy" ai slop clones Alan Watts, which is one of the recognizable slop if you know the landscape..

In [ ]:
import numpy as np

with stage("load Qwen3-TTS (narration)"):
    qwen_tts = Qwen3TTSModel.from_pretrained(
        QWEN_TTS_MODEL,
        device_map="cuda:0",
        dtype=torch.bfloat16,
        attn_implementation="sdpa",
    )

with stage("synthesise narration"):
    prompt_items = qwen_tts.create_voice_clone_prompt(
        ref_audio=str(ref_paths[NARRATOR_FILE]),
        ref_text=ref_texts[NARRATOR_FILE],
    )
    wavs, sr = qwen_tts.generate_voice_clone(
        text=sentences,
        language=["English"] * len(sentences),
        voice_clone_prompt=prompt_items,
    )

gap = np.zeros(int(SENTENCE_GAP_MS / 1000 * sr), dtype=np.float32)
narration = np.concatenate([np.concatenate([np.asarray(w, dtype=np.float32), gap])
                            for w in wavs])
sf.write(NARRATION, narration, sr)

narration_seconds = len(narration) / sr
print(f"{len(wavs)} sentences -> {NARRATION.name}: {narration_seconds:.1f}s at {sr} Hz")
display(Audio(str(NARRATION)))

In [ ]:
free_vram(qwen_tts)

### 2.3 — Word-level timings, and a look back at the previous lab

Whisper returns for the last time, now pointed at our own synthetic audio with `word_timestamps=True`. Two things come out of one transcription pass.

**A round-trip check.** We know exactly what the script said. Whisper tells us what it heard. The word error rate between them is a real intelligibility metric for the TTS output — no reference recording needed, no human listener, and it's the same measurement the previous lab was built around, pointed the other way.

**A one-word-per-cue SRT.** This is what the caption renderer eats. One word per cue is the format of the genre — it's a retention device, not a readability one — and

In [ ]:
with stage("load faster-whisper (timings)"):
    whisper = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="float16")

with stage("transcribe narration"):
    segments, info = whisper.transcribe(
        str(NARRATION), language="en", beam_size=5, word_timestamps=True,
    )
    segments = list(segments)

words = [w for seg in segments for w in (seg.words or [])]
heard = " ".join(w.word.strip() for w in words)
print(f"{len(segments)} segments, {len(words)} words with timings")

In [ ]:
PUNCT = re.compile(r"[^\w\s']")


def normalise(text: str) -> list:
    '''Lowercase, strip punctuation, split on whitespace.'''
    return PUNCT.sub(" ", text.lower()).split()


def wer(reference: str, hypothesis: str) -> float:
    '''Word error rate: Levenshtein distance over words, divided by reference length.'''
    ref, hyp = normalise(reference), normalise(hypothesis)
    previous = list(range(len(hyp) + 1))
    for i, r in enumerate(ref, start=1):
        current = [i]
        for j, h in enumerate(hyp, start=1):
            current.append(min(previous[j] + 1,          # deletion
                               current[j - 1] + 1,       # insertion
                               previous[j - 1] + (r != h)))  # substitution
        previous = current
    return previous[-1] / max(len(ref), 1)


asked = " ".join(sentences)
print("WE ASKED FOR:\n")
print(asked)
print("\nWHISPER HEARD:\n")
print(heard)
print(f"\nWER: {wer(asked, heard):.3f}  ({len(normalise(asked))} reference words)")

In [ ]:
from datetime import timedelta
import srt

cues = [
    srt.Subtitle(
        index=i,
        start=timedelta(seconds=w.start),
        end=timedelta(seconds=w.end),
        content=w.word.strip(),
    )
    for i, w in enumerate(words, start=1)
]

SRT_PATH.write_text(srt.compose(cues), encoding="utf-8")
print(f"{len(cues)} one-word cues -> {SRT_PATH.name}\n")
print(srt.compose(cues[:6]))

In [ ]:
free_vram(whisper)

### 2.4 — Compositing

No moviepy, no editing library. One `ffmpeg` call through `subprocess`, because everything we need is a flag.

Three decisions worth naming:

- **A random start offset into the gameplay footage.** Content farms do this so a thousand videos over the same clip don't look identical. We probe the background's actual duration with `ffprobe` rather than assuming it, and bound the offset so the slice fits.
- **The original gameplay audio is dropped.** We map video from input 0 and audio from input 1 and never mix.

In [ ]:
import random


def probe_duration(path) -> float:
    '''Container duration in seconds, via ffprobe.'''
    return float(run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                      "-of", "default=nw=1:nk=1", path]))


with stage("composite video"):
    narration_duration = probe_duration(NARRATION)
    background_duration = probe_duration(background_path)

    random.seed(SEED)
    offset = random.uniform(0.0, max(0.0, background_duration - narration_duration))
    print(f"narration {narration_duration:.1f}s | background {background_duration:.1f}s "
          f"| offset {offset:.1f}s")

    run([
        "ffmpeg", "-y",
        "-ss", f"{offset:.3f}", "-t", f"{narration_duration:.3f}", "-i", background_path,
        "-i", NARRATION,
        "-filter:v", (f"scale={VIDEO_W}:{VIDEO_H}:force_original_aspect_ratio=increase,"
                      f"crop={VIDEO_W}:{VIDEO_H}"),
        "-map", "0:v:0", "-map", "1:a:0",
        "-c:v", "libx264", "-preset", PRESET, "-crf", str(CRF), "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-b:a", AUDIO_BITRATE,
        "-shortest", "-movflags", "+faststart",
        COMPOSITE,
    ])

print(f"{COMPOSITE.name}: {COMPOSITE.stat().st_size / 1e6:.1f} MB, "
      f"{probe_duration(COMPOSITE):.1f}s")

### 2.5 — Captions

`beautiful-captions` takes our SRT, writes an ASS subtitle file, and burns it in with libass. We patched its imports and installed its fonts back in 0.5;

**We supply the SRT.** The library also has a `transcribe()` method that would generate one for us — via AssemblyAI, over the network, with an API key. We already have word timings from a model running on this machine. Never call it.

**`font` takes a file stem, not a font name.** Passing `"Montserrat"` does not raise. It writes `Fontname: None` into the ASS, libass falls back to DejaVu Sans, and your MrBeast video comes out looking like a LaTeX document.


In [ ]:
from beautiful_captions import Video as BCVideo, CaptionConfig

with stage("render captions"):
    BCVideo(str(COMPOSITE), config=CaptionConfig(
        style={"font": CAPTION_FONT,
               "font_size": CAPTION_FONT_SIZE,
               "color": CAPTION_COLOR,
               "outline_color": CAPTION_OUTLINE,
               "outline_thickness": CAPTION_THICKNESS,
               "verticle_position": CAPTION_POSITION,   # their spelling; 0.0 bottom -> 1.0 top
               "max_words_per_line": MAX_WORDS_PER_LINE,
               "auto_scale_font": True},
        animation={"enabled": True, "type": CAPTION_ANIMATION, "keyframes": CAPTION_KEYFRAMES},
        diarization={"enabled": False},
    )).add_captions(srt_input_path=str(SRT_PATH), output_path=str(FINAL))

print(f"{FINAL.name}: {FINAL.stat().st_size / 1e6:.1f} MB")

In [ ]:
ass_file = max(FINAL.parent.glob("*.ass"), key=lambda p: p.stat().st_mtime)
print(f"Generated ASS: {ass_file.name}\n")
for line in ass_file.read_text(encoding="utf-8", errors="replace").splitlines():
    if line.startswith("Style:"):
        print(line)

print("\nThe Fontname field above must NOT read 'None'.")
print(f"If it does, CAPTION_FONT ({CAPTION_FONT!r}) is not a valid file stem and libass "
      f"has silently fallen back to DejaVu Sans.")

### 2.6 — The result

Both a playable embed and a single extracted frame. The frame is not redundant: embedded video often fails to render in a notebook that's been closed and reopened, and I want the output visible in the saved `.ipynb` when this gets presented.

In [ ]:
from IPython.display import Video, Image

run(["ffmpeg", "-y", "-ss", "2", "-i", FINAL, "-frames:v", "1", "-q:v", "2", FRAME])

display(Image(str(FRAME), width=270))
display(Video(str(FINAL), embed=True, width=270))

---

## 3 — The slop timer

`stage()` has been recording every step. Here's the bill.

In [ ]:
for title in ["load faster-whisper (timings)", "load faster-whisper large-v3", "load script LLM", "load Qwen3-TTS"]:
  del TIMINGS[title]
total = sum(TIMINGS.values())
video_seconds = probe_duration(FINAL)

print(f"{'stage':<34}{'seconds':>10}{'share':>9}")
print("-" * 53)
for name, seconds in sorted(TIMINGS.items(), key=lambda kv: -kv[1]):
    print(f"{name:<34}{seconds:>10.1f}{seconds / total * 100:>8.1f}%")
print("-" * 53)
print(f"{'TOTAL':<34}{total:>10.1f}{100.0:>8.1f}%")

print(f"\nFinished video: {video_seconds:.1f}s")
print(f"Compute time:   {total:.1f}s")
print(f"\n{total / video_seconds:.1f} compute-seconds per second of finished video.")

## 4 — Outputs

In [ ]:
print("Files written:\n")
for directory in (REF_DIR, OUTPUT_DIR):
    print(f"{directory}/")
    for path in sorted(directory.iterdir()):
        if path.is_file():
            print(f"  {path.name:30} {path.stat().st_size:>12,} bytes")
    print()

print("Download from the file browser in the left sidebar, or:")
print(f"  from google.colab import files; files.download('{FINAL}')")

## 5 — Exercises

Each one changes a single variable. Run it, listen to or watch the difference, and write down what moved.

1. **Swap the narrator.** Change `NARRATOR_FILE` to a different Yae Miko clip and rerun Part 2 from 2.2. How much of the video's character came from the choice of reference? Is the effect bigger or smaller than you expected relative to everything else in the pipeline?

2. **Turn the temperature down.** Set `TEMPERATURE = 0.3`, regenerate the script, and diff it against the original. The slop is still slop. So where does the sloppiness actually live — in the writing, or in the format the writing is poured into?

---
# Lab Cooked by yours truly